In [1]:
!pip install psutil codecarbon onnxruntime-gpu segmentation-models-pytorch sklearn

  Using cached sklearn-0.0.post12.tar.gz (2.6 kB)
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'error'


  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> [15 lines of output]
      The 'sklearn' PyPI package is deprecated, use 'scikit-learn'
      rather than 'sklearn' for pip commands.
      
      Here is how to fix this error in the main use cases:
      - use 'pip install scikit-learn' rather than 'pip install sklearn'
      - replace 'sklearn' by 'scikit-learn' in your pip requirements files
        (requirements.txt, setup.py, setup.cfg, Pipfile, etc ...)
      - if the 'sklearn' package is used by one of your dependencies,
        it would be great if you take some time to track which package uses
        'sklearn' instead of 'scikit-learn' and report it to their issue tracker
      - as a last resort, set the environment variable
        SKLEARN_ALLOW_DEPRECATED_SKLEARN_PACKAGE_INSTALL=True to avoid this error
      
      More information is available at
      https://github.com/scikit-learn/sklearn-pypi-packag

In [3]:
import os
import time
import torch
import numpy as np
import pandas as pd
import psutil
import torch.nn as nn
import torch.nn.functional as F
import segmentation_models_pytorch as smp
import onnxruntime as ort
from tqdm import tqdm
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from codecarbon import EmissionsTracker

# ================= CONFIG =================
PTH_MODEL_PATH = r"E:\\FPT\\DAP391m\\model\\best_model_resnet101.pth"
ONNX_MODEL_PATH = "E:\\FPT\\DAP391m\\github\\Flood-Forecasting\\flood_resnet101_web_int8.onnx"
DATA_FOLDER = r"E:\\FPT\\DAP391m\\model\\outputres101\test"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TEST_FILES = [f for f in os.listdir(DATA_FOLDER) if f.endswith('.npz')][:50] # Benchmark trên 50 file

# ================= MODEL ARCHITECTURE =================
class ASPP(nn.Module):
    def __init__(self, in_dims, out_dims):
        super().__init__()
        self.conv1 = nn.Conv2d(in_dims, out_dims, 1, bias=False)
        self.conv2 = nn.Conv2d(in_dims, out_dims, 3, padding=6, dilation=6, bias=False)
        self.conv3 = nn.Conv2d(in_dims, out_dims, 3, padding=12, dilation=12, bias=False)
        self.conv_pool = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(in_dims, out_dims, 1, bias=False))
        self.fuse = nn.Sequential(nn.Conv2d(out_dims*4, out_dims, 1), nn.BatchNorm2d(out_dims), nn.ReLU())
    def forward(self, x):
        h, w = x.shape[-2:]
        res = [self.conv1(x), self.conv2(x), self.conv3(x),
               torch.nn.functional.interpolate(self.conv_pool(x), size=(h,w), mode='bilinear', align_corners=True)]
        return self.fuse(torch.cat(res, dim=1))

class FlexibleFloodModel(nn.Module):
    def __init__(self, encoder_name='resnet101'):
        super().__init__()
        self.base = smp.Unet(encoder_name=encoder_name, encoder_weights=None, in_channels=8, classes=1, decoder_attention_type='scse')
        ch = self.base.encoder.out_channels[-1]
        self.aspp = ASPP(ch, ch)
    def forward(self, x):
        feats = self.base.encoder(x); feats[-1] = self.aspp(feats[-1])
        dec = self.base.decoder(feats); return self.base.segmentation_head(dec)

# ================= CORE METRICS FUNCTION =================
def get_all_metrics(y_true, y_pred):
    mask = (y_true > 0.001) & (y_true < 0.999)
    if not np.any(mask): return [0]*6
    y_t, y_p = y_true[mask], y_pred[mask]
    
    mse = mean_squared_error(y_t, y_p)
    mae = mean_absolute_error(y_t, y_p)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_t, y_p)
    nse = 1 - (np.sum((y_t - y_p)**2) / (np.sum((y_t - np.mean(y_t))**2) + 1e-10))
    
    # KGE (Kling-Gupta Efficiency)
    r = np.corrcoef(y_t, y_p)[0, 1] if np.std(y_p) > 0 else 0
    alpha = np.std(y_p) / (np.std(y_t) + 1e-10)
    beta = np.mean(y_p) / (np.mean(y_t) + 1e-10)
    kge = 1 - np.sqrt((r-1)**2 + (alpha-1)**2 + (beta-1)**2)
    
    return nse, mse, mae, rmse, kge, r2

# ================= BENCHMARK ENGINE =================
def run_benchmark():
    final_stats = []

    # --- CHUẨN BỊ MÔ HÌNH ---
    # PyTorch
    model_pt = FlexibleFloodModel().to(DEVICE)
    ckpt = torch.load(PTH_MODEL_PATH, map_location=DEVICE, weights_only=False)
    model_pt.load_state_dict(ckpt['model_state_dict'] if 'model_state_dict' in ckpt else ckpt)
    model_pt.eval()
    
    # ONNX (Chạy trên CPU để giả lập môi trường Web thực tế)
    session = ort.InferenceSession(ONNX_MODEL_PATH, providers=['CPUExecutionProvider'])
    input_name = session.get_inputs()[0].name

    # Stats chuẩn hóa
    g_min = ckpt.get('g_min', np.zeros(8)).reshape(8,1,1)
    g_max = ckpt.get('g_max', np.ones(8)).reshape(8,1,1)

    for mode in ['PyTorch (FP32)', 'ONNX (INT8)']:
        print(f"\n🚀 Đang Benchmark hệ thống: {mode}...")
        
        tracker = EmissionsTracker(log_level="error", measure_power_secs=1)
        tracker.start()
        
        start_time = time.perf_counter()
        mem_start = psutil.Process().memory_info().rss / (1024**2)
        all_metrics = []

        for f in tqdm(TEST_FILES):
            data = np.load(os.path.join(DATA_FOLDER, f))
            y_true = data['y']
            
            if mode == 'PyTorch (FP32)':
                x = torch.from_numpy(data['x']).float().to(DEVICE).unsqueeze(0)
                x = (x - torch.tensor(g_min).to(DEVICE)) / (torch.tensor(g_max).to(DEVICE) - torch.tensor(g_min).to(DEVICE) + 1e-7)
                with torch.no_grad():
                    pred = torch.sigmoid(model_pt(x)).squeeze().cpu().numpy()
            else:
                x = (data['x'] - g_min) / (g_max - g_min + 1e-7)
                x = x[np.newaxis, ...].astype(np.float32)
                logits = session.run(None, {input_name: x})[0]
                pred = 1 / (1 + np.exp(-logits.squeeze()))
            
            all_metrics.append(get_all_metrics(y_true, pred))

        # Tính toán kết quả cuối
        duration = time.perf_counter() - start_time
        mem_end = psutil.Process().memory_info().rss / (1024**2)
        energy_consumed = tracker.stop() # kWh
        
        # Tính Watt Usage trung bình (Power = Energy / Time)
        # 1 kWh = 3,600,000 Joules. Power (W) = Joules / Seconds
        avg_watt = (energy_consumed * 3600000) / duration if duration > 0 else 0
        
        avg_m = np.mean(all_metrics, axis=0)
        file_size = os.path.getsize(PTH_MODEL_PATH if 'PyTorch' in mode else ONNX_MODEL_PATH) / (1024**2)

        final_stats.append({
            "Mode": mode,
            "NSE ↑": avg_m[0],
            "MSE ↓": avg_m[1],
            "MAE ↓": avg_m[2],
            "RMSE ↓": avg_m[3],
            "KGE ↑": avg_m[4],
            "R2 ↑": avg_m[5],
            "Model Size (MB)": file_size,
            "Speed (s/img)": duration / len(TEST_FILES),
            "RAM Usage (MB)": mem_end - mem_start,
            "Watt Usage (W)": avg_watt
        })

    # --- XUẤT BẢNG ---
    df = pd.DataFrame(final_stats).set_index("Mode").T
    print("\n" + "="*70)
    print("📊 BẢNG SO SÁNH CHỈ SỐ PERFORMANCE & HIỆU NĂNG HỆ THỐNG")
    print("="*70)
    pd.set_option('display.precision', 5)
    print(df)
    print("="*70)
    return df

if __name__ == "__main__":
    run_benchmark()

[codecarbon WARNING @ 22:19:52] Multiple instances of codecarbon are allowed to run at the same time.



🚀 Đang Benchmark hệ thống: PyTorch (FP32)...


100%|██████████| 50/50 [03:48<00:00,  4.58s/it]



🚀 Đang Benchmark hệ thống: ONNX (INT8)...


100%|██████████| 50/50 [02:02<00:00,  2.45s/it]



📊 BẢNG SO SÁNH CHỈ SỐ PERFORMANCE & HIỆU NĂNG HỆ THỐNG
Mode             PyTorch (FP32)  ONNX (INT8)
NSE ↑                   0.49511      0.49367
MSE ↓                   0.02455      0.02465
MAE ↓                   0.12140      0.12149
RMSE ↓                  0.15223      0.15260
KGE ↑                   0.73730      0.73763
R2 ↑                    0.49511      0.49367
Model Size (MB)      1757.63043    146.73119
Speed (s/img)           4.57910      2.44783
RAM Usage (MB)       2683.59766  -1628.37109
Watt Usage (W)         22.72123     19.40864
